# Module 02 — Lesson 08
# File Handling (TXT, CSV, JSON)

---

> All the data you'll ever analyse lives in a file. This lesson teaches you to open it.

Until now, every value in your programs has been typed by hand. Real data science doesn't work that way. A dataset arrives as a CSV on your desktop, an API response saved as JSON, or a plain text log from a server. Before Pandas, NumPy, or any ML library can help you — you need to read the file.

In this lesson you'll load data using only Python's built-in `csv`, `json`, and `os` modules. This is intentionally the hard way. By the end you'll feel the pain of doing it manually — and Module 5 (Pandas) will feel like a superpower the moment you try it.

**What you'll build:**
A complete file-based pipeline: read a messy CSV → clean it → compute statistics → write a filtered output CSV and a plain-text summary report.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Open, read, and write text files using the `with` statement and correct file modes
- Always specify `encoding="utf-8"` and understand why it matters
- Use `csv.reader` and `csv.DictReader` to parse CSV files correctly
- Write structured data back to CSV with `csv.writer` and `csv.DictWriter`
- Read and write JSON using `json.load()` / `json.dump()`
- Build a complete file-based data pipeline: load → clean → analyse → save
- Handle file errors gracefully using try/except

---
## 1. Opening Files — `open()` and the `with` Statement

Every file operation in Python starts with `open()`. The two things that matter most:

| Parameter | What it does |
|---|---|
| `mode` | `"r"` read · `"w"` write (create/overwrite) · `"a"` append · `"rb"`/`"wb"` binary |
| `encoding` | Always pass `"utf-8"` — without it, Python uses your OS default, which varies |

### The `with` statement

Always open files with `with`. It guarantees the file is closed even if an exception occurs. A file left open wastes system resources and can lock the file on Windows.

In [ ]:
import os
import csv
import json

# Write a file first so we have something to read
sample_text = """Student Report — Batch 2024
Name: Priya Sharma
Score: 88
City: Mumbai
"""

with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("File written.")
print(f"File exists: {os.path.exists('sample.txt')}")
print(f"File size  : {os.path.getsize('sample.txt')} bytes")

### 1.1 Reading Modes

In [ ]:
# Mode 1: f.read() — entire file as one string
with open("sample.txt", "r", encoding="utf-8") as f:
    content = f.read()
print("f.read():")
print(repr(content[:60]), "...")

In [ ]:
# Mode 2: f.readlines() — list of lines (includes \n)
with open("sample.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
print("f.readlines():")
for i, line in enumerate(lines):
    print(f"  [{i}] {line!r}")

In [ ]:
# Mode 3: iterate line by line — best for large files (memory efficient)
print("Iterate line by line (stripped):")
with open("sample.txt", "r", encoding="utf-8") as f:
    for line in f:
        stripped = line.strip()
        if stripped:            # skip blank lines
            print(f"  {stripped}")

### 1.2 Writing Modes

In [ ]:
# "w" mode — creates the file if it doesn't exist; OVERWRITES if it does
with open("log.txt", "w", encoding="utf-8") as f:
    f.write("Session started\n")
    f.write("Loaded 100 records\n")

# "a" mode — appends to the file; creates it if it doesn't exist
with open("log.txt", "a", encoding="utf-8") as f:
    f.write("Processing complete\n")
    f.write("3 errors found\n")

# Read back to verify
with open("log.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# f.writelines() — write a list of strings (you must add \n yourself)
entries = ["Priya: 88", "Rohan: 72", "Meera: 95"]

with open("scores.txt", "w", encoding="utf-8") as f:
    f.writelines(line + "\n" for line in entries)

with open("scores.txt", "r", encoding="utf-8") as f:
    print(f.read())

---
## 2. Working with File Paths

Hard-coding paths like `"C:\\Users\\Priya\\data.csv"` breaks on every other machine. Use `os.path` to build portable paths.

In [ ]:
import os

# os.path.join builds the correct separator for the OS (/ on Mac/Linux, \ on Windows)
data_dir  = os.path.join("data")
file_path = os.path.join("data", "students.csv")
print(f"data dir  : {data_dir}")
print(f"file path : {file_path}")

# Useful os.path helpers
print(f"\nexists    : {os.path.exists(file_path)}")
print(f"dirname   : {os.path.dirname(file_path)}")
print(f"basename  : {os.path.basename(file_path)}")
name, ext = os.path.splitext(file_path)
print(f"name      : {name}")
print(f"extension : {ext}")

In [ ]:
# Create a directory safely — exist_ok=True means no error if it already exists
os.makedirs("output", exist_ok=True)
print(f"output/ exists: {os.path.isdir('output')}")

# List files in a directory
print(f"\nFiles in data/:")
for filename in os.listdir("data"):
    fp = os.path.join("data", filename)
    print(f"  {filename:<25} ({os.path.getsize(fp)} bytes)")

---
## 3. Reading CSV Files

CSV (Comma-Separated Values) is the most common data format you'll encounter. Never parse it with `.split(",")` — that breaks the moment a value contains a comma inside quotes.

Python's `csv` module handles all the edge cases for you.

In [ ]:
# Why .split(",") is wrong — this is a real CSV quirk
messy_line = '"Sharma, Priya",20,Mumbai,CS,88'

print("split(','):")
print(f"  {messy_line.split(',')}")
# ['"Sharma', ' Priya"', '20', 'Mumbai', 'CS', '88'] — 6 fields instead of 5!

print("\ncsv.reader (correct):")
import csv
import io
reader = csv.reader(io.StringIO(messy_line))
print(f"  {next(reader)}")
# ['Sharma, Priya', '20', 'Mumbai', 'CS', '88'] — 5 fields, correct

### 3.1 `csv.reader` — Row as List

In [ ]:
import csv

CSV_PATH = os.path.join("data", "students.csv")

rows = []
with open(CSV_PATH, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)       # first row is the header
    for row in reader:
        rows.append(row)

print(f"Header : {header}")
print(f"Total rows: {len(rows)}")
print(f"\nFirst 3 rows:")
for row in rows[:3]:
    print(f"  {row}")

In [ ]:
# With csv.reader, you must access fields by index — error-prone and hard to read
print("Positional access (fragile):")
for row in rows[:3]:
    name  = row[0]    # what if columns change order?
    score = row[4]
    print(f"  {name}: {score}")

### 3.2 `csv.DictReader` — Row as Dictionary (recommended)

In [ ]:
# DictReader uses the header row as keys — much safer and clearer
records = []
with open(CSV_PATH, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(dict(row))   # each row is already an OrderedDict

print(f"Records loaded: {len(records)}")
print(f"\nFirst record:")
for key, val in records[0].items():
    print(f"  {key:<8}: {val!r}")

print(f"\nNamed access (safe):")
for r in records[:3]:
    print(f"  {r['name']}: score={r['score']!r}")

### 3.3 Type Conversion — Everything Is a String

CSV stores everything as text. `DictReader` gives you `"88"` not `88`. You must convert types yourself — and handle the dirty values.

In [ ]:
def safe_float(value, default=None):
    """Convert to float, returning default on empty or non-numeric strings."""
    if value is None or str(value).strip() == "":
        return default
    try:
        return float(value)
    except (ValueError, TypeError):
        return default

def safe_int(value, default=None):
    if value is None or str(value).strip() == "":
        return default
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return default


# Show the raw vs converted difference
print(f"{'Name':<16} {'Raw score':<12} {'Converted':<12} {'Type'}")
print("─" * 52)
for r in records[:6]:
    raw  = r["score"]
    conv = safe_float(raw)
    print(f"  {r['name']:<14} {str(raw):<12} {str(conv):<12} {type(conv).__name__}")

In [ ]:
# Parse the full dataset — convert types and flag missing values
def parse_student(raw):
    """Clean and type-convert a raw CSV row dict."""
    return {
        "name"   : raw["name"].strip().title(),
        "age"    : safe_int(raw["age"]),
        "city"   : raw["city"].strip().title(),
        "dept"   : raw["dept"].strip().upper(),
        "score"  : safe_float(raw["score"]),
        "valid"  : safe_float(raw["score"]) is not None,
    }

students = [parse_student(r) for r in records]

valid   = [s for s in students if s["valid"]]
invalid = [s for s in students if not s["valid"]]

print(f"Total: {len(students)} | Valid score: {len(valid)} | Missing/invalid: {len(invalid)}")
print(f"\nRows with missing scores:")
for s in invalid:
    print(f"  {s['name']} (score={s['score']!r})")

---
## 4. Writing CSV Files

### 4.1 `csv.writer` — Write Rows as Lists

In [ ]:
# Build a summary table to write
dept_stats = {}
for s in valid:
    dept = s["dept"]
    if dept not in dept_stats:
        dept_stats[dept] = []
    dept_stats[dept].append(s["score"])

summary_rows = []
for dept, scores in sorted(dept_stats.items()):
    avg = sum(scores) / len(scores)
    summary_rows.append([dept, len(scores), round(avg, 1), min(scores), max(scores)])


# Write with csv.writer
out_path = os.path.join("output", "dept_summary.csv")
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["dept", "count", "avg_score", "min_score", "max_score"])  # header
    writer.writerows(summary_rows)

print(f"Written: {out_path}")

# Verify by reading back
with open(out_path, "r", encoding="utf-8") as f:
    print(f.read())

### 4.2 `csv.DictWriter` — Write Rows as Dicts (recommended)

In [ ]:
# Write the cleaned student records — only those with valid scores
FIELDS = ["name", "age", "city", "dept", "score", "grade"]

def add_grade(student):
    s = student["score"]
    grade = "A" if s >= 80 else "B" if s >= 60 else "C" if s >= 40 else "F"
    return {**student, "grade": grade}

graded = [add_grade(s) for s in valid]


out_path = os.path.join("output", "students_clean.csv")
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
    writer.writeheader()            # writes the header row from fieldnames
    writer.writerows(graded)        # each dict row, fields matched by name

print(f"Written: {out_path} ({len(graded)} rows)")

# Preview the first 5 rows
with open(out_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print(f"\n{'Name':<16} {'Dept':<5} {'Score':>6} {'Grade'}")
    print("─" * 35)
    for row in list(reader)[:5]:
        print(f"  {row['name']:<14} {row['dept']:<5} {float(row['score']):>6.1f} {row['grade']}")

---
## 5. JSON Files

JSON is the standard format for API responses, config files, and nested data. Use the `json` module — never try to build JSON strings manually.

In [ ]:
import json

# Writing JSON — json.dump()
config = {
    "app": "Student Analytics v1.0",
    "settings": {
        "passing_score": 40,
        "grade_thresholds": {"A": 80, "B": 60, "C": 40},
        "valid_depts": ["CS", "ME", "IT", "EE"],
    },
    "output_dir": "output",
}

config_path = os.path.join("output", "config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)   # indent=2 makes it human-readable

print(f"Written: {config_path}")

# Check what it looks like on disk
with open(config_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Reading JSON — json.load()
with open(config_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)

# It's a regular Python dict — access normally
print(f"App        : {loaded['app']}")
print(f"Pass mark  : {loaded['settings']['passing_score']}")
print(f"Valid depts: {loaded['settings']['valid_depts']}")
print(f"Grade A    : {loaded['settings']['grade_thresholds']['A']}+")

In [ ]:
# json.dumps() / json.loads() — work with strings instead of files
# Useful for API responses or sending data over a network

student_dict = {"name": "Priya Sharma", "score": 88, "dept": "CS"}

# Python dict → JSON string
json_string = json.dumps(student_dict)
print(f"JSON string: {json_string!r}")
print(f"Type: {type(json_string)}")

# JSON string → Python dict
back_to_dict = json.loads(json_string)
print(f"\nBack to dict: {back_to_dict}")
print(f"score + 10  : {back_to_dict['score'] + 10}")

In [ ]:
# JSON type mapping — Python ↔ JSON
mapping = """
  Python         JSON
  ──────────     ────────
  dict       →   object  {}
  list       →   array   []
  str        →   string  ""
  int/float  →   number
  True/False →   true/false
  None       →   null
"""
print(mapping)

# Write department stats as JSON (list of dicts)
dept_report = [
    {"dept": dept, "count": len(scores),
     "avg": round(sum(scores)/len(scores), 1),
     "pass_rate": round(sum(1 for s in scores if s >= 40) / len(scores) * 100, 1)}
    for dept, scores in sorted(dept_stats.items())
]

report_path = os.path.join("output", "dept_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(dept_report, f, indent=2)

print(f"Written: {report_path}")
with open(report_path, "r", encoding="utf-8") as f:
    print(f.read())

---
## 6. Exception Handling for Files

File operations fail for predictable reasons: the file doesn't exist, you don't have permission, or the encoding is wrong. Always handle these gracefully.

In [ ]:
# The three file exceptions you'll meet most
def safe_read_file(path, encoding="utf-8"):
    """
    Read a text file safely.
    Returns (content, error) — exactly one will be None.
    """
    try:
        with open(path, "r", encoding=encoding) as f:
            return f.read(), None
    except FileNotFoundError:
        return None, f"File not found: {path}"
    except PermissionError:
        return None, f"Permission denied: {path}"
    except UnicodeDecodeError as e:
        return None, f"Encoding error (try a different encoding): {e}"


# Test all three cases
cases = [
    "sample.txt",           # exists
    "no_such_file.txt",     # doesn't exist
    "data/students.csv",    # exists but let's verify
]

for path in cases:
    content, error = safe_read_file(path)
    if error:
        print(f"  ✗ {error}")
    else:
        first_line = content.split("\n")[0]
        print(f"  ✓ {path} — first line: {first_line!r}")

In [ ]:
# Safe CSV loader — returns empty list + error message instead of crashing
def load_csv(path):
    """
    Load a CSV file using DictReader.
    Returns (records, error).
    """
    try:
        with open(path, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames is None:
                return [], "File appears to be empty"
            return [dict(row) for row in reader], None
    except FileNotFoundError:
        return [], f"File not found: {path}"
    except Exception as e:
        return [], f"Unexpected error: {e}"


data, err = load_csv("data/students.csv")
print(f"Loaded: {len(data)} rows, error: {err!r}")

data2, err2 = load_csv("data/missing.csv")
print(f"Loaded: {len(data2)} rows, error: {err2!r}")

---
## 7. Applied — Complete File Pipeline

This is where everything comes together. A real data pipeline: load from file → clean → analyse → write two output files.

In [ ]:
# ── Step 1: Load ──────────────────────────────────────────────────────────────
raw_records, err = load_csv(os.path.join("data", "students.csv"))
if err:
    raise RuntimeError(f"Cannot load data: {err}")
print(f"Step 1 — Loaded {len(raw_records)} raw records")


# ── Step 2: Clean & Validate ─────────────────────────────────────────────────
def clean_record(raw, row_num):
    """
    Clean and validate one raw CSV row.
    Returns (clean_dict, error_string_or_None).
    """
    name  = str(raw.get("name") or "").strip().title()
    if not name:
        return None, f"Row {row_num}: name is blank"

    age   = safe_int(raw.get("age"))
    city  = str(raw.get("city") or "").strip().title()
    dept  = str(raw.get("dept") or "").strip().upper()
    score = safe_float(raw.get("score"))

    if score is None:
        return None, f"Row {row_num}: {name!r} has missing/invalid score"
    if not (0 <= score <= 100):
        return None, f"Row {row_num}: {name!r} score {score} out of range"

    grade = "A" if score >= 80 else "B" if score >= 60 else "C" if score >= 40 else "F"
    return {"name": name, "age": age, "city": city, "dept": dept,
            "score": score, "grade": grade}, None


clean_records = []
error_log     = []
for i, raw in enumerate(raw_records):
    cleaned, error = clean_record(raw, i + 2)  # +2 because row 1 = header
    if cleaned:
        clean_records.append(cleaned)
    else:
        error_log.append(error)

print(f"Step 2 — {len(clean_records)} clean | {len(error_log)} errors")
print(f"  Errors: {error_log}")

In [ ]:
# ── Step 3: Analyse ───────────────────────────────────────────────────────────
def compute_stats(values):
    if not values:
        return {}
    n      = len(values)
    mean   = sum(values) / n
    sorted_v = sorted(values)
    mid    = n // 2
    median = sorted_v[mid] if n % 2 else (sorted_v[mid-1] + sorted_v[mid]) / 2
    variance = sum((x - mean) ** 2 for x in values) / n
    std    = variance ** 0.5
    return {"count": n, "mean": round(mean, 1), "median": round(median, 1),
            "std": round(std, 1), "min": min(values), "max": max(values)}


all_scores = [s["score"] for s in clean_records]
overall    = compute_stats(all_scores)

# Dept breakdown
dept_groups = {}
for s in clean_records:
    dept_groups.setdefault(s["dept"], []).append(s["score"])
dept_analysis = {d: compute_stats(sc) for d, sc in sorted(dept_groups.items())}

# Grade distribution
grade_count = {}
for s in clean_records:
    grade_count[s["grade"]] = grade_count.get(s["grade"], 0) + 1

print("Step 3 — Analysis complete")
print(f"  Overall mean: {overall['mean']}, pass rate: "
      f"{sum(1 for x in all_scores if x >= 40)/len(all_scores)*100:.0f}%")

In [ ]:
# ── Step 4: Write output CSV ─────────────────────────────────────────────────
out_csv = os.path.join("output", "pipeline_output.csv")

with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age", "city", "dept", "score", "grade"])
    writer.writeheader()
    writer.writerows(clean_records)

print(f"Step 4 — Wrote {len(clean_records)} rows → {out_csv}")


# ── Step 5: Write text summary report ────────────────────────────────────────
report_lines = [
    "=" * 50,
    "STUDENT ANALYTICS REPORT",
    "=" * 50,
    "",
    f"Total records loaded  : {len(raw_records)}",
    f"Clean records         : {len(clean_records)}",
    f"Records with errors   : {len(error_log)}",
    "",
    "OVERALL STATISTICS",
    "-" * 30,
    f"Count   : {overall['count']}",
    f"Mean    : {overall['mean']}",
    f"Median  : {overall['median']}",
    f"Std Dev : {overall['std']}",
    f"Min     : {overall['min']}",
    f"Max     : {overall['max']}",
    f"Pass rate (>=40): {sum(1 for x in all_scores if x >= 40)/len(all_scores)*100:.0f}%",
    "",
    "BY DEPARTMENT",
    "-" * 30,
]

for dept, stats in dept_analysis.items():
    report_lines.append(
        f"  {dept:<4}  n={stats['count']}  mean={stats['mean']}  "
        f"min={stats['min']}  max={stats['max']}"
    )

report_lines += ["", "GRADE DISTRIBUTION", "-" * 30]
total = len(clean_records)
for grade in ["A", "B", "C", "F"]:
    count = grade_count.get(grade, 0)
    bar   = "█" * int(count / total * 20)
    report_lines.append(f"  {grade}: {count:>3}  {count/total*100:>5.1f}%  {bar}")

report_lines += ["", "ERRORS ENCOUNTERED", "-" * 30]
for err in error_log:
    report_lines.append(f"  {err}")

report_lines.append("")
report_lines.append("=" * 50)

report_path = os.path.join("output", "summary_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Step 5 — Report written → {report_path}")

# Print the report
with open(report_path, "r", encoding="utf-8") as f:
    print()
    print(f.read())

---
## ⚠️ Common Mistakes

In [ ]:
# ── Mistake 1: Not using 'with' — file might stay open ───────────────────────

# ❌ If an exception occurs between open() and close(), the file is never closed
# f = open("data.csv", "r")
# data = f.read()    # if this raises, f.close() never runs
# f.close()

# ✅ 'with' guarantees close() even on exception
# with open("data.csv", "r", encoding="utf-8") as f:
#     data = f.read()

print("Mistake 1: Always use 'with open(...) as f' — not open()/close() manually.")

In [ ]:
# ── Mistake 2: 'w' mode overwrites silently ───────────────────────────────────

# Write some data
with open("important.txt", "w", encoding="utf-8") as f:
    f.write("Important data!\nDo not lose this.\n")

# ❌ 'w' DESTROYS the file contents — no warning
with open("important.txt", "w", encoding="utf-8") as f:
    f.write("Oops — everything above is gone.\n")

with open("important.txt", "r", encoding="utf-8") as f:
    print(f"After 'w' overwrite: {f.read()!r}")

# ✅ If you want to ADD to existing content, use 'a' (append)
with open("important.txt", "a", encoding="utf-8") as f:
    f.write("But this line is safely appended.\n")

with open("important.txt", "r", encoding="utf-8") as f:
    print(f"After 'a' append: {f.read()!r}")

In [ ]:
# ── Mistake 3: Forgetting newline='' in csv.writer ────────────────────────────

# ❌ Without newline='', Python adds \r\n on Windows, causing extra blank rows
# with open("out.csv", "w", encoding="utf-8") as f:   # missing newline=''
#     writer = csv.writer(f)
#     writer.writerow(["a", "b"])

# ✅ Always pass newline='' to csv.writer and csv.DictWriter
# with open("out.csv", "w", newline="", encoding="utf-8") as f:
#     writer = csv.writer(f)
#     writer.writerow(["a", "b"])

print("Mistake 3: Always use newline='' when opening files for csv.writer/DictWriter.")

In [ ]:
# ── Mistake 4: Missing encoding — works on your machine, breaks elsewhere ─────

# ❌ Default encoding varies: UTF-8 on Mac/Linux, CP1252 on Windows India locale
# with open("file.txt", "r") as f:              # no encoding
#     content = f.read()                         # works on your machine...

# ✅ Always be explicit
# with open("file.txt", "r", encoding="utf-8") as f:
#     content = f.read()

print("Mistake 4: Always pass encoding='utf-8' to open(). No exceptions.")

In [ ]:
# ── Mistake 5: f.write() needs \n — it doesn't add one automatically ─────────

with open("no_newlines.txt", "w", encoding="utf-8") as f:
    f.write("line one")
    f.write("line two")      # joins onto the same line!
    f.write("line three")

with open("no_newlines.txt", "r", encoding="utf-8") as f:
    print(f"Without \\n: {f.read()!r}")

with open("no_newlines.txt", "w", encoding="utf-8") as f:
    f.write("line one\n")
    f.write("line two\n")
    f.write("line three\n")

with open("no_newlines.txt", "r", encoding="utf-8") as f:
    print(f"With \\n   : {f.read()!r}")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Student Log Writer

Write a program that:
1. Accepts a list of student dicts (name, score)
2. Writes each student's result as a line to `output/results.txt` — e.g. `"Priya Sharma: 88 — PASS"`
3. Reads the file back and prints it
4. Appends a summary line at the end: `"Total: 5 students | Pass: 4 | Fail: 1"`

In [ ]:
students_data = [
    {"name": "Priya Sharma",  "score": 88},
    {"name": "Rohan Verma",   "score": 38},
    {"name": "Meera Nair",    "score": 95},
    {"name": "Karan Mehta",   "score": 61},
    {"name": "Ananya Singh",  "score": 22},
]

# Your code here
# Step 1: write results to output/results.txt

# Step 2: read it back and print

# Step 3: append the summary line

### Exercise 2 — CSV Stats Calculator

Load `data/students.csv` using `csv.DictReader`. Compute and print:
- Total rows, valid rows (non-empty score), invalid rows
- Mean, min, max score (valid rows only)
- Count per department (sorted alphabetically)
- Count per city (top 3 cities by count)

Handle missing/invalid scores with your `safe_float()` function.

In [ ]:
# Your code here
CSV_PATH = os.path.join("data", "students.csv")

### Exercise 3 — Filter and Save

Load `data/students.csv`, filter to only the passing students (score ≥ 40), and save them to `output/passing_students.csv`. The output CSV should include an extra `grade` column (`A`/`B`/`C`).

Also write a `output/failing_students.csv` for those who failed or had missing scores.

In [ ]:
# Your code here
# Hint: use csv.DictReader to load, csv.DictWriter to write
# Make sure to include newline='' in open() for the writers

### Exercise 4 — JSON Config Read / Write

Write `save_config(config_dict, path)` and `load_config(path)` functions. Then:
1. Create a config dict with at least 3 nested keys
2. Save it to `output/my_config.json`
3. Load it back and verify two specific values match what you saved
4. Update one value and save again — verify the old file is replaced

In [ ]:
def save_config(config_dict, path):
    # Your code here
    pass

def load_config(path):
    # Your code here
    pass

# Test
my_config = {
    # Build your own config here
}

### Exercise 5 — (Challenge) Full Mini Pipeline

Build a complete `run_pipeline(input_csv, output_dir)` function that:
1. Loads the CSV safely (returns early with an error message if file not found)
2. Cleans each row (type convert, validate, collect errors)
3. Writes three output files:
   - `clean_data.csv` — valid records with grade column
   - `error_log.csv` — rows that failed, with an `error` column describing why
   - `summary.json` — overall stats + dept breakdown + grade distribution
4. Prints a one-line summary: `"Pipeline complete: 23/25 rows clean, 2 errors. Files saved to output/"`

Test it with `data/students.csv`.

In [ ]:
def run_pipeline(input_csv, output_dir):
    # Step 1: Load

    # Step 2: Clean

    # Step 3a: Write clean_data.csv

    # Step 3b: Write error_log.csv

    # Step 3c: Write summary.json

    # Step 4: Print summary
    pass


run_pipeline(os.path.join("data", "students.csv"), "output")

---
## 📌 Key Takeaways

- **Always use `with open(...) as f` and always pass `encoding="utf-8"`.** These two rules together prevent 90% of file-handling bugs: the `with` block guarantees the file closes even on exception, and explicit encoding prevents the script from breaking on machines with different locale settings.

- **Use `csv.DictReader` / `csv.DictWriter`, not `csv.reader` / `.split(",")`.** `DictReader` gives you field names as keys — your code stays readable when columns change order. Never parse CSV with `.split(",")`: it silently breaks on any value that contains a comma.

- **Everything from a CSV is a string — type-convert explicitly and handle failures.** A CSV row never gives you an integer or float automatically. Write a `safe_float()` / `safe_int()` helper, use it on every numeric column, and handle the `None` cases. This discipline is the foundation of the data cleaning module (Module 6).

---
## What's Next?

**Module 02 Project — Student Management System & Expense Tracker**  
You now have the complete Python toolkit: variables, operators, loops, functions, lists, dictionaries, exception handling, and file I/O. The module project puts all of it together into two real programs — a student management system that persists data to CSV, and an expense tracker that saves and loads JSON. After the project, Module 3 (Mathematics for Data Science) builds the statistical intuition behind everything ML.